In [1]:
import psycopg2

# ─── Configuration ───────────────────────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "database": "sec_filings",
    "user": "ashish",
    "password": "ashish"
}

In [2]:
# ─── Create table ────────────────────────────────────────────────────────────
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

cur.execute("""
    DROP TABLE IF EXISTS filing_chunks;
    
    CREATE TABLE filing_chunks (
        id              SERIAL PRIMARY KEY,
        ticker          VARCHAR(10) NOT NULL,
        filing_type     VARCHAR(10) NOT NULL,
        filing_date     DATE,
        section         TEXT,
        chunk_index     INTEGER,
        chunk_text      TEXT NOT NULL,
        embedding       vector(384),
        source_file     TEXT,
        created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Index for fast vector search (IVFFlat)
    -- We'll create this AFTER inserting all data (faster bulk insert)
    -- CREATE INDEX ON filing_chunks USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);

    -- Indexes for filtering
    CREATE INDEX idx_ticker ON filing_chunks(ticker);
    CREATE INDEX idx_filing_type ON filing_chunks(filing_type);
    CREATE INDEX idx_filing_date ON filing_chunks(filing_date);
""")

conn.commit()
cur.close()
conn.close()

In [3]:
print("Table 'filing_chunks' created successfully!")
print("Schema:")
print("  id, ticker, filing_type, filing_date, section, chunk_index,")
print("  chunk_text, embedding(384), source_file, created_at")

Table 'filing_chunks' created successfully!
Schema:
  id, ticker, filing_type, filing_date, section, chunk_index,
  chunk_text, embedding(384), source_file, created_at
